In [ ]:
!pip install transformers datasets torch torchaudio librosa captum

In [ ]:
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset
import torch
import librosa

import torch.nn.functional as F
from captum.attr import IntegratedGradients

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
TARGET_IDX = 60

In [ ]:
model_id = "VasilisAsim/hubert-finetuned-Ravdess"
model_ft = AutoModelForAudioClassification.from_pretrained(model_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model_ft.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft.to(device)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

dataset = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))

dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx)
})

test_dataset = dataset["test"]
print(f"Μέγεθος test split: {len(test_dataset)} δείγματα")

In [ ]:
#Wrapper function για την μετατροπή της εξόδου σε tensors
def forward_func(input_values):
    outputs = model_ft(input_values)
    return F.softmax(outputs.logits, dim=-1)

In [ ]:
ig = IntegratedGradients(forward_func)

In [ ]:
import os
import numpy as np
from scipy.interpolate import interp1d

def resample_attribution(attr, target_frames=50):
    attr = np.abs(attr)
    x_original = np.linspace(0, 1, len(attr))
    x_target = np.linspace(0, 1, target_frames)
    f = interp1d(x_original, attr, kind='linear')
    resampled = f(x_target)
    resampled = resampled / (resampled.sum() + 1e-10)
    return resampled

save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/03_hubert_ravdess/attributions"   # ✅
os.makedirs(save_dir, exist_ok=True)

In [ ]:
def explain_sample(idx, split="test"):
    sample = dataset[split][idx]
    speech = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    if sr != feature_extractor.sampling_rate:
        speech = librosa.resample(speech, orig_sr=sr, target_sr=feature_extractor.sampling_rate)

    inputs = feature_extractor(speech, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt")
    input_values = inputs["input_values"].to(device).requires_grad_()

    true_label = model_ft.config.id2label[sample["labels"]]

    with torch.no_grad():
        pred_class = forward_func(input_values).argmax(dim=-1).item()

    baseline = torch.zeros_like(input_values)
    attributions, delta = ig.attribute(
        input_values, baselines=baseline, target=pred_class,
        n_steps=50, return_convergence_delta=True
    )

    return speech, attributions.squeeze().detach().cpu().numpy(), pred_class, true_label, delta.item()

In [ ]:
speech, attr, pred_class, true_label, delta = explain_sample(TARGET_IDX, split="test")

attr_abs = np.abs(attr)
window = 400
attr_smooth = np.convolve(attr_abs, np.ones(window)/window, mode='same')

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(speech, color='gray')
axes[0].set_title(f"Κυματομορφή — True label: {true_label}")

axes[1].plot(attr_smooth, color='red')
axes[1].set_title(f"Integrated Gradients — Predicted: {model_ft.config.id2label[pred_class]}")
plt.tight_layout()
plt.show()

resampled_ig = resample_attribution(attr, target_frames=50)
np.save(f"{save_dir}/IG_idx{TARGET_IDX}.npy", resampled_ig)
print(f"Αποθηκεύτηκε: IG_idx{TARGET_IDX}.npy")